# Extracción y limpieza de la tabla maestra de productos

**Proyecto:** Análisis de ventas de la empresa TIMARAN  
**Objetivo del notebook:** construir una tabla maestra limpia de productos a partir del archivo original de ventas.  
**Resultado esperado:** archivo Excel `Tabla_Maestra_Productos.xlsx`, listo para usar en Power BI, Python u otros procesos de análisis.

---

## Flujo general del proceso

1. Cargar el archivo de ventas original.
2. Extraer las columnas relacionadas con productos y clasificaciones.
3. Estandarizar nombres de columnas y textos.
4. Separar el código del producto y el nombre del producto.
5. Limpiar espacios innecesarios.
6. Validar valores nulos y categorías.
7. Reemplazar valores faltantes en clasificaciones.
8. Exportar la tabla maestra final.

## 1. Importación de librerías

En esta sección se cargan las librerías necesarias para el proceso.

`pandas` se utiliza para leer, transformar y exportar la información.  
`Path` permite manejar rutas de archivos de forma más segura y compatible entre Windows, Linux y macOS.

In [43]:
from pathlib import Path

import pandas as pd
from IPython.display import display

## 2. Configuración de rutas

Aquí se define la carpeta donde se encuentra el archivo original y donde se guardará el resultado final.

La ruta se construye con `Path` para evitar problemas con las barras invertidas `\` de Windows.

In [44]:
# Carpeta base donde están los archivos del proyecto
DATA_DIR = Path("../Data")

# Archivo de entrada
ARCHIVO_ENTRADA = DATA_DIR / "Informacion sin limpiar.xlsx"

# Archivo de salida
ARCHIVO_SALIDA = DATA_DIR / "Tabla_Maestra_Productos.xlsx"

# Hoja del archivo Excel que contiene la información de ventas
HOJA_VENTAS = "Análisis de ventas"

print(f"Archivo de entrada: {ARCHIVO_ENTRADA}")
print(f"Archivo de salida: {ARCHIVO_SALIDA}")

Archivo de entrada: ..\Data\Informacion sin limpiar.xlsx
Archivo de salida: ..\Data\Tabla_Maestra_Productos.xlsx


## 3. Carga del archivo original

Se carga el archivo Excel original usando la hoja `Análisis de ventas`.

El parámetro `header=3` indica que los encabezados reales están en la cuarta fila del archivo Excel, porque Python empieza a contar desde cero.  
El parámetro `index_col=0` toma la primera columna como índice temporal, y luego se reinicia el índice para dejarla nuevamente como columna normal.

In [45]:
df_sin_limpiar = pd.read_excel(
    ARCHIVO_ENTRADA,
    sheet_name=HOJA_VENTAS,
    header=3,
    index_col=0
)

# Regresar el índice como columna normal
df_sin_limpiar = df_sin_limpiar.reset_index()

print(f"Filas cargadas: {df_sin_limpiar.shape[0]}")
print(f"Columnas cargadas: {df_sin_limpiar.shape[1]}")

display(df_sin_limpiar.head())

Filas cargadas: 9842
Columnas cargadas: 5


,PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA ...,ENVASE,SADMAN,105 ML,54
1,[2610] ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,[2608] ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NaN,NaN
4,[2278] A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 4. Extracción de columnas de productos

La tabla original contiene más información de la necesaria para la tabla maestra.

Para este proceso se toman únicamente las primeras cinco columnas, que corresponden al producto y sus clasificaciones principales.

In [46]:
df_productos = df_sin_limpiar.iloc[:, 0:5].copy()

print(f"Filas: {df_productos.shape[0]}")
print(f"Columnas: {df_productos.shape[1]}")

display(df_productos.head())

Filas: 9842
Columnas: 5


,PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA ...,ENVASE,SADMAN,105 ML,54
1,[2610] ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,[2608] ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NaN,NaN
4,[2278] A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 5. Estandarización de nombres de columnas

Se convierten los encabezados a mayúsculas para trabajar con nombres uniformes y evitar errores por diferencias de escritura.

In [47]:
df_productos.columns = (
    df_productos.columns
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

display(df_productos.head())

,PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA ...,ENVASE,SADMAN,105 ML,54
1,[2610] ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,[2608] ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NaN,NaN
4,[2278] A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 6. Validación de columnas esperadas

Antes de continuar, se valida que el archivo tenga las columnas necesarias.

Esta validación ayuda a detectar cambios en el formato del archivo original antes de que el proceso genere errores más adelante.

In [48]:
columnas_esperadas = [
    "PRODUCTO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV"
]

columnas_faltantes = [col for col in columnas_esperadas if col not in df_productos.columns]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas en el archivo: {columnas_faltantes}")

print("Validación correcta: todas las columnas esperadas están disponibles.")

Validación correcta: todas las columnas esperadas están disponibles.


## 7. Conversión de textos a mayúsculas

Se convierten los valores de texto a mayúsculas para evitar duplicados falsos.

Por ejemplo, `aseo`, `Aseo` y `ASEO` quedarían como una sola categoría: `ASEO`.

In [49]:
columnas_texto = [
    "PRODUCTO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV"
]

for columna in columnas_texto:
    df_productos[columna] = df_productos[columna].astype("string").str.upper()

display(df_productos.head())

,PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA ...,ENVASE,SADMAN,105 ML,54
1,[2610] ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,[2608] ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,<NA>,<NA>
4,[2278] A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 8. Separación del código y nombre del producto

En la columna `PRODUCTO` el código viene dentro de corchetes, por ejemplo:

`[12345] NOMBRE DEL PRODUCTO`

Con esta transformación se crean dos columnas independientes:

- `CODIGO_PRODUCTO`: código del producto.
- `NOMBRE_PRODUCTO`: nombre limpio del producto.

In [50]:
df_productos["CODIGO_PRODUCTO"] = (
    df_productos["PRODUCTO"]
    .str.extract(r"\[(.*?)\]", expand=False)
    .astype("string")
    .str.strip()
)

df_productos["NOMBRE_PRODUCTO"] = (
    df_productos["PRODUCTO"]
    .str.replace(r"\[.*?\]\s*", "", regex=True)
    .str.strip()
)

# Se elimina la columna original porque ya fue separada en código y nombre
df_productos = df_productos.drop(columns=["PRODUCTO"])

# Se reorganizan las columnas para dejar una estructura más clara
df_productos = df_productos[
    [
        "CODIGO_PRODUCTO",
        "NOMBRE_PRODUCTO",
        "CLASIFICACION I",
        "CLASIFICACION II",
        "CLASIFICACION III",
        "CLASIFICACION IV"
    ]
]

display(df_productos.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,<NA>,<NA>
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 9. Limpieza de espacios

Se eliminan espacios al inicio, al final y espacios dobles dentro de los textos.

Esto es importante porque valores como `ASEO`, ` ASEO` y `ASEO  GENERAL` pueden afectar los conteos, filtros y relaciones en Power BI.

In [51]:
columnas_limpiar = [
    "NOMBRE_PRODUCTO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV"
]

df_productos[columnas_limpiar] = df_productos[columnas_limpiar].apply(
    lambda col: col.str.strip().str.replace(r"\s+", " ", regex=True)
)

df_productos["CODIGO_PRODUCTO"] = (
    df_productos["CODIGO_PRODUCTO"]
    .astype("string")
    .str.replace(r"\s+", "", regex=True)
)

display(df_productos.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,<NA>,<NA>
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 10. Revisión exploratoria de clasificaciones

Estas consultas permiten revisar qué valores existen en las clasificaciones.

Sirven para detectar nombres mal escritos, categorías repetidas, espacios ocultos o valores que deberían corregirse manualmente en la tabla base.

In [52]:
print("Valores únicos en CLASIFICACION I:")
display(df_productos["CLASIFICACION I"].dropna().sort_values().unique())

print("Conteo de valores en CLASIFICACION III:")
display(df_productos["CLASIFICACION III"].value_counts(dropna=False))

Valores únicos en CLASIFICACION I:


<StringArray>
[ 'ACEITE', 'ALCOHOL',    'CAJA',   'CREMA',  'ENVASE', 'ESENCIA', 'FIJADOR',
   'FUNDA', 'GLITTER',  'MALETA', 'MAQUINA',    'STCK',    'TAPA', 'VALVULA']
Length: 14, dtype: string

Conteo de valores en CLASIFICACION III:


CLASIFICACION III
<NA>             3769
30 ML             673
100 ML            502
50 ML             451
60 ML             355
                 ... 
DAVID BECKHAM       4
EMMIR               4
JUICY COUTURE       4
JOOP                4
MARIE FARINA        4
Name: count, Length: 203, dtype: Int64

## 11. Creación de la tabla maestra

Se crea una copia del DataFrame limpio para dejarlo como tabla maestra de productos.

Esta tabla será la fuente principal para relacionar los productos con sus clasificaciones.

In [53]:
df_productos_maestro = df_productos.copy()

df_productos_maestro = df_productos_maestro[
    [
        "CODIGO_PRODUCTO",
        "NOMBRE_PRODUCTO",
        "CLASIFICACION I",
        "CLASIFICACION II",
        "CLASIFICACION III",
        "CLASIFICACION IV"
    ]
]

display(df_productos_maestro.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,<NA>,<NA>
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 12. Validación de valores nulos

Se revisa cuántos valores nulos existen en cada columna.

Esta revisión permite decidir qué columnas deben corregirse o completarse antes de exportar el archivo final.

In [54]:
display(df_productos_maestro.isnull().sum())

CODIGO_PRODUCTO         0
NOMBRE_PRODUCTO         0
CLASIFICACION I         0
CLASIFICACION II     3463
CLASIFICACION III    3769
CLASIFICACION IV     3952
dtype: int64

## 13. Tratamiento de clasificaciones vacías

En las columnas de clasificación secundaria se reemplazan los valores vacíos o nulos por `NO APLICA`.

Esto ayuda a evitar problemas en Power BI al momento de crear segmentaciones, filtros o relaciones.

In [55]:
columnas_rellenar = [
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV"
]

# Convertir espacios vacíos en valores nulos reales
df_productos_maestro[columnas_rellenar] = (
    df_productos_maestro[columnas_rellenar]
    .replace(r"^\s*$", pd.NA, regex=True)
)

# Reemplazar nulos por NO APLICA
df_productos_maestro[columnas_rellenar] = (
    df_productos_maestro[columnas_rellenar]
    .fillna("NO APLICA")
)

display(df_productos_maestro.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 14. Validación después del reemplazo

Se vuelve a revisar la cantidad de valores nulos para confirmar que las clasificaciones secundarias fueron completadas correctamente.

In [56]:
display(df_productos_maestro.isnull().sum())

CODIGO_PRODUCTO      0
NOMBRE_PRODUCTO      0
CLASIFICACION I      0
CLASIFICACION II     0
CLASIFICACION III    0
CLASIFICACION IV     0
dtype: int64

## 15. Revisión de frecuencias por clasificación

Estos conteos ayudan a entender la distribución de productos por clasificación.

También permiten identificar si existen muchas filas con `NO APLICA`, lo cual puede indicar que hay información incompleta en el archivo original.

In [57]:
print("Conteo CLASIFICACION II:")
display(df_productos_maestro["CLASIFICACION II"].value_counts(dropna=False))

print("Conteo CLASIFICACION III:")
display(df_productos_maestro["CLASIFICACION III"].value_counts(dropna=False))

print("Conteo CLASIFICACION IV:")
display(df_productos_maestro["CLASIFICACION IV"].value_counts(dropna=False))

Conteo CLASIFICACION II:


CLASIFICACION II
NO APLICA          3463
ESENCIA GENERAL    2092
ESENCIA NICHO      1313
CILINDRO            263
BALA                162
                   ... 
SIENNA                4
FRASCO                4
PROBADOR              4
EMIR                  4
ROMA                  3
Name: count, Length: 101, dtype: Int64

Conteo CLASIFICACION III:


CLASIFICACION III
NO APLICA        3769
30 ML             673
100 ML            502
50 ML             451
60 ML             355
                 ... 
DAVID BECKHAM       4
EMMIR               4
JUICY COUTURE       4
JOOP                4
MARIE FARINA        4
Name: count, Length: 203, dtype: Int64

Conteo CLASIFICACION IV:


CLASIFICACION IV
NO APLICA    3952
DM           1384
HM           1171
UNISEX        902
120           503
108           285
1             279
160           180
72            164
48            152
144            80
99             66
60             60
70             56
100            56
54             48
180            48
80             48
40             48
96             47
84             39
112            31
168            29
NEOPET         20
64             20
90             20
105            17
50             16
117            13
240            12
140             9
300             8
CREMA           8
FIJADOR         8
104             7
143             7
ACEITE          4
ALCOHOL         4
282             4
66              4
480             4
110             4
176             4
264             4
135             4
1600            4
1800            4
200             4
204             1
Name: count, dtype: Int64

## 16. Revisión de productos con clasificaciones incompletas

Se muestran algunos productos donde una o más clasificaciones quedaron como `NO APLICA`.

Esta revisión sirve para validar manualmente si realmente no aplica o si la clasificación debe ser corregida desde el archivo fuente.

In [58]:
productos_sin_clasificacion_completa = df_productos_maestro[
    (df_productos_maestro["CLASIFICACION II"] == "NO APLICA") |
    (df_productos_maestro["CLASIFICACION III"] == "NO APLICA") |
    (df_productos_maestro["CLASIFICACION IV"] == "NO APLICA")
]

print(f"Productos con alguna clasificación en NO APLICA: {productos_sin_clasificacion_completa.shape[0]}")

display(productos_sin_clasificacion_completa.head(20))

Productos con alguna clasificación en NO APLICA: 4055


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
58,1425,BALA ALUMINIO DORADA X 300 UND,ENVASE,BALA ALUMINIO,NO APLICA,300
59,1424,BALA ALUMINIO NEGRA X 282 UND,ENVASE,BALA ALUMINIO,NO APLICA,282
60,0020,BALA ALUMINIO PLATEADA X 300 UND,ENVASE,BALA ALUMINIO,NO APLICA,300
62,2670,BAMBU FINAL,ESENCIA,ESENCIA INDUSTRIAL,DIFUSORES,NO APLICA
102,2449,BOQUILLA MAQUINA GRAFADORA # 15,MAQUINA,NO APLICA,NO APLICA,NO APLICA
103,2228,BOQUILLA MAQUINA GRAFADORA # 18,MAQUINA,NO APLICA,NO APLICA,NO APLICA
104,2227,BOQUILLA MAQUINA GRAFADORA # 20,MAQUINA,NO APLICA,NO APLICA,NO APLICA
125,2782,CAJA CILINDRICA BLANCA 10 ML X 539 UND,CAJA,CILINDRICA,10 ML,NO APLICA
126,2780,CAJA CILINDRICA BLANCA 100 ML X 120 UND,CAJA,CILINDRICA,100 ML,NO APLICA


## 17. Eliminación de duplicados

Como esta tabla será una tabla maestra, cada código de producto debería aparecer una sola vez.

Si existen códigos repetidos, se conserva el primer registro encontrado.

In [59]:
filas_antes = df_productos_maestro.shape[0]

df_productos_maestro = df_productos_maestro.drop_duplicates(
    subset=["CODIGO_PRODUCTO"],
    keep="first"
).reset_index(drop=True)

filas_despues = df_productos_maestro.shape[0]

print(f"Filas antes de eliminar duplicados: {filas_antes}")
print(f"Filas después de eliminar duplicados: {filas_despues}")
print(f"Duplicados eliminados: {filas_antes - filas_despues}")

Filas antes de eliminar duplicados: 9842
Filas después de eliminar duplicados: 2502
Duplicados eliminados: 7340


## 18. Validación final de la tabla maestra

Se revisan dimensiones, primeras filas y nulos finales antes de exportar.

Esta sección funciona como control de calidad del resultado.

In [60]:
print("Dimensiones finales de la tabla maestra:")
print(f"Filas: {df_productos_maestro.shape[0]}")
print(f"Columnas: {df_productos_maestro.shape[1]}")

print("\nValores nulos finales:")
display(df_productos_maestro.isnull().sum())

display(df_productos_maestro.head())

Dimensiones finales de la tabla maestra:
Filas: 2502
Columnas: 6

Valores nulos finales:


CODIGO_PRODUCTO      0
NOMBRE_PRODUCTO      0
CLASIFICACION I      0
CLASIFICACION II     0
CLASIFICACION III    0
CLASIFICACION IV     0
dtype: int64

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


## 19. Normalizacion de codigos de productos

Se realiza normalizacion con el fin de que todos lo codigos queden con 4 digitos 

In [61]:
# ============================================================
# FUNCIÓN PARA NORMALIZAR CÓDIGOS DE PRODUCTO
# ============================================================

def normalizar_codigo_producto(df, columna_codigo="CODIGO_PRODUCTO", largo_codigo=4):
    """
    Convierte los códigos de producto a texto, elimina espacios,
    quita decimales innecesarios y agrega ceros a la izquierda.
    
    Ejemplo:
    1      -> 0001
    58     -> 0058
    495    -> 0495
    2608   -> 2608
    0001   -> 0001
    """
    
    df = df.copy()
    
    df[columna_codigo] = (
        df[columna_codigo]
        .astype("string")
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(largo_codigo)
    )
    
    return df

In [62]:
df_productos_maestro_final = normalizar_codigo_producto(
    df_productos_maestro,
    columna_codigo="CODIGO_PRODUCTO",
    largo_codigo=4
)

## 20. Exportación a Excel

Finalmente se exporta la tabla maestra limpia a un archivo Excel.

Este archivo queda listo para ser usado en Power BI como tabla de dimensiones o tabla maestra de productos.

In [63]:
# Crear la carpeta Data si no existe
DATA_DIR.mkdir(parents=True, exist_ok=True)

df_productos_maestro_final.to_excel(
    ARCHIVO_SALIDA,
    sheet_name="Productos",
    index=False
)

print(f"Archivo exportado correctamente en: {ARCHIVO_SALIDA}")

Archivo exportado correctamente en: ..\Data\Tabla_Maestra_Productos.xlsx


## 20. Resultado final

Al ejecutar todo el notebook se genera el archivo:

`Data/Tabla_Maestra_Productos.xlsx`

Este archivo contiene la tabla maestra de productos limpia, estandarizada y lista para integrarse con la tabla de ventas.